In [ ]:
import bz2
import pandas as pd
import pickle
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
# get mapSolarSystems.jsonl from the data folder
solar_systems_df = pd.read_json("data/mapSolarSystems.jsonl", lines=True)
solar_systems_df = solar_systems_df[
    [
        '_key', 
        'border', 
        'constellationID', 
        'hub', 
        'international',
        'luminosity', 
        'name', 
        'planetIDs', 
        'position', 
        'position2D', 
        'radius',
        'regionID', 
        'regional', 
        'securityClass', 
        'securityStatus', 
        'starID',
        'stargateIDs', 
        'corridor', 
        'fringe', 
        'wormholeClassID', 
        'visualEffect',
        'disallowedAnchorCategories', 
        'disallowedAnchorGroups', 
        'factionID'
    ]
]
solar_systems_df["name"] = [x["en"] for x in solar_systems_df["name"]]
solar_systems_df

In [ ]:
# get mapStargates.jsonl from the data folder
stargates_df = pd.read_json("data/mapStargates.jsonl", lines=True)
stargates_df = stargates_df[['_key', 'destination', 'position', 'solarSystemID', 'typeID']]
stargates_df

In [ ]:
system_id_dict = solar_systems_df[["name", "_key"]].set_index("name")["_key"].to_dict()
system_id_dict

In [ ]:
system_name_dict = solar_systems_df[["_key", "name"]].set_index("_key")["name"].to_dict()
system_name_dict

In [ ]:
# create a graph of the stargates
G = nx.Graph()
for _, row in stargates_df[stargates_df["position"].notna()].iterrows():
    G.add_node(system_name_dict[row['solarSystemID']], position=row['position'])
    G.add_edge(system_name_dict[row['solarSystemID']], system_name_dict[row['destination']["solarSystemID"]])
list(G.edges)[:5]

In [ ]:
# calculate shortest path between two systems
source_system = "Kino"
target_system = "Arvasaras"
shortest_path = nx.shortest_path(G, source=source_system, target=target_system)
shortest_path

In [ ]:
component = next(c for c in nx.connected_components(G) if "Jita" in c)
subgraph = G.subgraph(component)

In [ ]:
route = nx.approximation.traveling_salesman_problem(
    subgraph, 
    # method=nx.approximation.greedy_tsp,
    method=nx.approximation.christofides,
    nodes=list(subgraph.nodes),
    cycle=True
    )
route

In [ ]:
i = route.index("Jita")
route = route[i:-1] + route[:i] + ["Jita"]

In [ ]:
route

In [ ]:
# save route to data/route.txt
with open("data/route.txt", "w") as f:
    for system in route:
        f.write(system + "\n")

In [ ]:
# read the route from the file
with open("data/route.txt", "r") as f:
    route = [line.strip() for line in f.readlines()]
route

In [ ]:
print(len(route))

In [ ]:
import base64
import json
import time
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlencode, urlparse, parse_qs

import requests


In [ ]:
# Create these keys for yourself by registering an application on the ESI developer portal: https://developers.eveonline.com/applications
id_path = ".venv/keys/client_id.txt"
secret_path = ".venv/keys/client_secret.txt"
with open(id_path, "r") as f:
    CLIENT_ID = f.read().strip()
with open(secret_path, "r") as f:
    CLIENT_SECRET = f.read().strip()

In [ ]:

CALLBACK_URL = "http://localhost:8080/callback"

In [ ]:

SCOPES = ["esi-ui.write_waypoint.v1"]

AUTH_URL = "https://login.eveonline.com/v2/oauth/authorize"
TOKEN_URL = "https://login.eveonline.com/v2/oauth/token"
ESI_BASE = "https://esi.evetech.net/latest"


def get_auth_code():
    auth_params = {
        "response_type": "code",
        "redirect_uri": CALLBACK_URL,
        "client_id": CLIENT_ID,
        "scope": " ".join(SCOPES),
        "state": "route-tool",
    }

    url = f"{AUTH_URL}?{urlencode(auth_params)}"
    print("Opening browser for EVE login...")
    webbrowser.open(url)

    result = {}

    class CallbackHandler(BaseHTTPRequestHandler):
        def do_GET(self):
            query = parse_qs(urlparse(self.path).query)

            if "code" in query:
                result["code"] = query["code"][0]
                self.send_response(200)
                self.end_headers()
                self.wfile.write(b"Login complete. You can close this tab.")
            else:
                self.send_response(400)
                self.end_headers()
                self.wfile.write(b"No authorization code found.")

        def log_message(self, format, *args):
            return

    server = HTTPServer(("localhost", 8080), CallbackHandler)
    server.handle_request()

    return result["code"]


def exchange_code_for_token(code):
    basic_auth = base64.b64encode(
        f"{CLIENT_ID}:{CLIENT_SECRET}".encode()
    ).decode()

    headers = {
        "Authorization": f"Basic {basic_auth}",
        "Content-Type": "application/x-www-form-urlencoded",
        "Host": "login.eveonline.com",
    }

    data = {
        "grant_type": "authorization_code",
        "code": code,
    }

    response = requests.post(TOKEN_URL, headers=headers, data=data)
    response.raise_for_status()
    return response.json()["access_token"]


def set_waypoints(access_token, system_ids):
    headers = {
        "Authorization": f"Bearer {access_token}",
        "User-Agent": "eve-route-tool/1.0",
    }

    for index, system_id in enumerate(system_ids):
        params = {
            "destination_id": system_id,
            "add_to_beginning": "false",
            "clear_other_waypoints": "true" if index == 0 else "false",
            "datasource": "tranquility",
        }

        response = requests.post(
            f"{ESI_BASE}/ui/autopilot/waypoint/",
            headers=headers,
            params=params,
        )

        if response.status_code != 204:
            raise RuntimeError(
                f"Failed setting waypoint {system_id}: "
                f"{response.status_code} {response.text}"
            )

        print(f"Added waypoint: {system_id}")
        time.sleep(0.25)




In [ ]:
# divide the route into chunks of 200 systems and set waypoints for each chunk
len(route)/200

In [ ]:
# Pick a chunk of 200 systems and set waypoints for that chunk, starting at 0.
i = 0
SYSTEMS_IDS = [system_id_dict[system] for system in route[i*200:(i+1)*200]]
SYSTEMS_IDS

In [ ]:
# Drop duplicates and keep the first occurrence of each system
seen = set()
unique_system_ids = []
for system_id in SYSTEMS_IDS:
    if system_id not in seen:
        unique_system_ids.append(system_id)
        seen.add(system_id)
unique_system_ids

In [ ]:

auth_code = get_auth_code()
access_token = exchange_code_for_token(auth_code)

In [ ]:
set_waypoints(access_token, unique_system_ids)
print("Route created in EVE.")